# Desenvolvimento residual por grupo e alvo

Esta rodada testa se modelos conseguem corrigir a média exponencial de 3 meses (EWMA3), especialmente no grupo alto. Em cada fold, o grupo alto é definido separadamente para CRIME, FURTO e ROUBO: top 5%, 10% ou 20% dos H3 pela contagem histórica daquele alvo no treino. 2022 não entra como linha de treino; pode fornecer histórico anterior. O período jul/2025–jun/2026 permanece reservado.

No grupo alto, treina modelos Ridge, Random Forest e HistGradientBoosting para prever o resíduo `risco - EWMA3`; a previsão final é `EWMA3 + resíduo previsto`, limitada a `[0,1]`. No grupo baixo, permanece o hurdle binomial negativo. A avaliação compara EWMA3 global, média histórica fixa e híbridos. O ganho contra EWMA3 é a métrica principal; ganhos positivos indicam que o modelo aprendeu uma correção útil.

Os folds são os mesmos 3 períodos expansivos de seis meses. Os grupos, a escala do score e o treino são recalculados dentro de cada fold. Execute a célula para gerar métricas, previsões OOF e atribuições dos grupos para cada alvo e limiar.


In [1]:
import numpy as np
import pandas as pd
import statsmodels.api as sm
from pathlib import Path
from sklearn.ensemble import RandomForestRegressor, HistGradientBoostingRegressor
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression, Ridge
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error

src=Path('../datasets/database_processed_h3_risk.csv')
if not src.exists(): src=Path('datasets/database_processed_h3_risk.csv')
out=src.resolve().parent
summary_path=out/'validation_target_specific_residual_h3_summary.csv'
fold_path=out/'validation_target_specific_residual_h3_fold_metrics.csv'
pred_path=out/'validation_target_specific_residual_h3_predictions.csv'
group_path=out/'validation_target_specific_residual_h3_assignments.csv'

START,END='2023-01-01','2025-06-01'
TARGETS={'RISCO_CRIME':'CNT_CRIME_COUNT','RISCO_FURTO':'FURTO_COUNT','RISCO_ROUBO':'ROUBO_COUNT'}
FEATURES={
'RISCO_CRIME':['crime_media_12m_log','crime_razao_3_12','crime_sazonal_12','crime_pct_meses_12m','crime_viz_rel','C_IPVS','pop_total','media_moradores','pct_jovens_15_29','taxa_alfabetizacao','pct_negros','pct_dom_casa','pct_dom_apartamento','pct_dom_cortico','MES_SIN','MES_COS','SEM_DOMICILIOS'],
'RISCO_FURTO':['furto_media_12m_log','furto_razao_3_12','furto_sazonal_12','furto_pct_meses_12m','furto_viz_rel','C_IPVS','pop_total','media_moradores','pct_jovens_15_29','taxa_alfabetizacao','pct_negros','pct_dom_casa','pct_dom_apartamento','pct_dom_cortico','MES_SIN','MES_COS','SEM_DOMICILIOS'],
'RISCO_ROUBO':['roubo_media_12m_log','roubo_razao_3_12','roubo_sazonal_12','roubo_viz_rel','C_IPVS','pop_total','media_moradores','pct_jovens_15_29','taxa_alfabetizacao','pct_negros','pct_dom_casa','pct_dom_apartamento','pct_dom_cortico','MES_SIN','MES_COS','SEM_DOMICILIOS']}
THRESHOLDS=[.05,.10,.20]
RF_PARAMS=dict(n_estimators=300,max_depth=12,min_samples_leaf=5,max_features=.7,bootstrap=True,max_samples=.8,n_jobs=-1,random_state=42)
HGB_PARAMS=dict(loss='squared_error',learning_rate=.05,max_iter=100,max_leaf_nodes=15,min_samples_leaf=30,l2_regularization=1.0,early_stopping=False,random_state=42)

D=pd.read_csv(src,parse_dates=['DATA'],low_memory=False).sort_values(['H3','DATA']).reset_index(drop=True)
D=D[D.DATA.between('2022-01-01','2026-06-01')].copy()
E=D[D.DATA.between(START,END)].sort_values(['DATA','H3']).reset_index(drop=True)
months=np.sort(E.DATA.unique()); folds=[]
for k in range(3,0,-1):
 end=len(months)-6*(k-1); start=end-6
 folds.append((np.flatnonzero(E.DATA.isin(months[:start])),np.flatnonzero(E.DATA.isin(months[start:end])),months[start:end]))

def score(c,K): return c.clip(lower=0)/(c.clip(lower=0)+K)
def metric(y,p): return {'R2':r2_score(y,p),'MAE':mean_absolute_error(y,p),'RMSE':float(np.sqrt(mean_squared_error(y,p)))}
def design(a,b,cols):
 X=a[cols].replace([np.inf,-np.inf],np.nan); Z=b[cols].replace([np.inf,-np.inf],np.nan)
 med=X.median().fillna(0); return X.fillna(med),Z.fillna(med)

metric_rows=[]; prediction_rows=[]; group_rows=[]
for fold,(itr,iva,val_dates) in enumerate(folds,1):
 train=E.iloc[itr].copy(); valid=E.iloc[iva].copy()
 for target,count_col in TARGETS.items():
  pos=train.loc[train[count_col]>0,count_col]; K=float(pos.quantile(.90)) if len(pos) else 1.
  # Fold-specific target score and its one-step-ahead EWMA/residual history.
  hist=D[['H3','DATA',count_col]].copy(); hist['y']=score(hist[count_col],K)
  hist['ewm3']=hist.groupby('H3',sort=False).y.transform(lambda s:s.shift(1).ewm(halflife=3,adjust=False,min_periods=1).mean())
  hist['residual']=hist.y-hist.ewm3
  hist['resid_lag1']=hist.groupby('H3',sort=False).residual.shift(1)
  hist['resid_mean3']=hist.groupby('H3',sort=False).residual.transform(lambda s:s.shift(1).rolling(3,min_periods=1).mean())
  extra=hist[['H3','DATA','y','ewm3','resid_lag1','resid_mean3']]
  tr=train.merge(extra,on=['H3','DATA'],how='left',validate='one_to_one')
  va=valid.merge(extra,on=['H3','DATA'],how='left',validate='one_to_one')
  target_features=FEATURES[target]
  residual_features=target_features+['ewm3','resid_lag1','resid_mean3']
  yva=va.y.to_numpy()
  # Overall EWM3 and fixed historical mean baselines.
  cell_mean=tr.groupby('H3').y.mean()
  predictions={'ewm3_global':va.ewm3.to_numpy(),
               'media_historica_fixa':va.H3.map(cell_mean).fillna(float(tr.y.mean())).to_numpy()}

  for share in THRESHOLDS:
   totals=tr.groupby('H3')[count_col].sum().sort_values(ascending=False,kind='stable')
   nh=max(1,int(np.ceil(share*len(totals))))
   high=set(totals.head(nh).index)
   group=pd.Series(np.where(totals.index.isin(high),'alta','baixa'),index=totals.index)
   tr['grupo']=tr.H3.map(group).fillna('baixa'); va['grupo']=va.H3.map(group).fillna('baixa')
   group_rows.append(pd.DataFrame({'fold':fold,'target':target,'share_high':share,'H3':totals.index,
      'grupo':group.values,'historico_contagem_alvo':totals.values,'inicio_validacao':pd.Timestamp(val_dates[0]),'fim_validacao':pd.Timestamp(val_dates[-1])}))
   hi=tr[tr.grupo.eq('alta')].copy(); lo=tr[tr.grupo.eq('baixa')].copy()
   vhi=va[va.grupo.eq('alta')].copy(); vlo=va[va.grupo.eq('baixa')].copy()

   # Low-group hurdle NB, trained using this target's low-activity cells only.
   event=(lo[count_col]>0).astype('int8'); Xlo,Xv_all=design(lo,va,target_features)
   classifier=make_pipeline(StandardScaler(),LogisticRegression(C=1.,max_iter=2000,random_state=42)).fit(Xlo,event)
   p_event=classifier.predict_proba(Xv_all)[:,1]
   positive=event.eq(1); pos_counts=lo.loc[positive,count_col].astype(float)
   mu=float(pos_counts.mean()); var=float(pos_counts.var()); alpha=max((var-mu)/(mu*mu),.01) if mu>0 else .01
   scaler=StandardScaler().fit(Xlo)
   Xpos=sm.add_constant(scaler.transform(Xlo.loc[positive]),has_constant='add')
   Xv=sm.add_constant(scaler.transform(Xv_all),has_constant='add')
   nb=sm.GLM(pos_counts.to_numpy(),Xpos,family=sm.families.NegativeBinomial(alpha=alpha)).fit(maxiter=100,disp=0)
   expected=p_event*np.maximum(nb.predict(Xv),0); low_score_all=expected/(expected+K); low_score=low_score_all[va.grupo.eq('baixa').to_numpy()]

   # High-group models predict residuals relative to EWM3.
   Xhi,Xvhi=design(hi,vhi,residual_features)
   residual_train=hi.y-hi.ewm3
   residual_models={
     'ridge':make_pipeline(StandardScaler(),Ridge(alpha=10.)),
     'rf':RandomForestRegressor(**RF_PARAMS),
     'hgb':HistGradientBoostingRegressor(**HGB_PARAMS)}
   high_scores={}
   for name,model in residual_models.items():
     model.fit(Xhi,residual_train)
     high_scores[name]=np.clip(vhi.ewm3.to_numpy()+model.predict(Xvhi),0,1)
   # Baseline hybrid: no residual correction for high, hurdle NB for low.
   hybrid_ewm=np.where(va.grupo.eq('alta'),va.ewm3.to_numpy(),low_score_all)
   predictions[f'ewm3alta_hurdlebaixa_top{int(share*100)}']=hybrid_ewm
   for model_name,high_pred in high_scores.items():
     combined=np.empty(len(va),dtype=float); is_hi=va.grupo.eq('alta').to_numpy(); combined[is_hi]=high_pred; combined[~is_hi]=low_score
     predictions[f'{model_name}resid_hurdle_top{int(share*100)}']=combined

   # Metrics for shared baselines, each hybrid, and high-only residual corrections.
   segments=[('todos',np.ones(len(va),bool)),('alta',va.grupo.eq('alta').to_numpy()),('baixa',va.grupo.eq('baixa').to_numpy())]
   model_preds={'ewm3_global':va.ewm3.to_numpy(),'media_historica_fixa':va.H3.map(cell_mean).fillna(float(tr.y.mean())).to_numpy(),
                f'ewm3alta_hurdlebaixa_top{int(share*100)}':hybrid_ewm}
   for name,vals in high_scores.items():
     full=np.array(low_score_all,copy=True); full[va.grupo.eq('alta').to_numpy()]=vals; model_preds[f'{name}resid_hurdle_top{int(share*100)}']=full
   for segment,mask in segments:
     if not mask.any(): continue
     for name,values in model_preds.items():
       mr=metric(yva[mask],values[mask])
       metric_rows.append({'fold':fold,'target':target,'share_high':share,'segmento':segment,'modelo':name,'n':int(mask.sum()),'K_fold':K,**mr})
     if segment=='alta':
       for name,values in high_scores.items():
         mr=metric(vhi.y.to_numpy(),values)
         metric_rows.append({'fold':fold,'target':target,'share_high':share,'segmento':'alta_somente','modelo':f'{name}_residual_sem_hurdle','n':len(vhi),'K_fold':K,**mr})
   # Store detailed OOF predictions per threshold.
   pf=pd.DataFrame({'fold':fold,'target':target,'share_high':share,'H3':va.H3.to_numpy(),'DATA':va.DATA.to_numpy(),
      'grupo':va.grupo.to_numpy(),'K_fold':K,'y_true':yva,'pred_ewm3_global':va.ewm3.to_numpy(),
      'pred_media_historica_fixa':va.H3.map(cell_mean).fillna(float(tr.y.mean())).to_numpy(),
      'pred_ewm3_alta_hurdle_baixa':hybrid_ewm})
   for name,high_pred in high_scores.items():
     comb=np.array(low_score_all,copy=True); comb[va.grupo.eq('alta').to_numpy()]=high_pred
     pf[f'pred_{name}_resid_hurdle']=comb
   prediction_rows.append(pf)

metrics=pd.DataFrame(metric_rows); preds=pd.concat(prediction_rows,ignore_index=True); groups=pd.concat(group_rows,ignore_index=True)
base=metrics[metrics.modelo.eq('ewm3_global')][['fold','target','share_high','segmento','RMSE']].rename(columns={'RMSE':'RMSE_ewm3'})
fixed=metrics[metrics.modelo.eq('media_historica_fixa')][['fold','target','share_high','segmento','RMSE']].rename(columns={'RMSE':'RMSE_mean'})
comp=metrics.merge(base,on=['fold','target','share_high','segmento'],how='left').merge(fixed,on=['fold','target','share_high','segmento'],how='left')
comp['gain_vs_ewm3_pct']=(comp.RMSE_ewm3-comp.RMSE)/comp.RMSE_ewm3*100
comp['gain_vs_mean_pct']=(comp.RMSE_mean-comp.RMSE)/comp.RMSE_mean*100
summary=(comp.groupby(['target','share_high','segmento','modelo'],as_index=False)
 .agg(folds=('fold','nunique'),RMSE_mean=('RMSE','mean'),MAE_mean=('MAE','mean'),R2_mean=('R2','mean'),
   gain_vs_ewm3_pct=('gain_vs_ewm3_pct','mean'),gain_vs_mean_pct=('gain_vs_mean_pct','mean'),
   folds_better_than_ewm3=('gain_vs_ewm3_pct',lambda s:int((s>0).sum()))))
metrics.to_csv(fold_path,index=False); summary.to_csv(summary_path,index=False); preds.to_csv(pred_path,index=False); groups.to_csv(group_path,index=False)
print('Overall results: positive gain_vs_ewm3 means residual model beats EWM3:')
print(summary[(summary.segmento=='todos')].sort_values(['target','RMSE_mean']).to_string(index=False))
print('High-group residual results:')
print(summary[summary.segmento=='alta_somente'].sort_values(['target','share_high','RMSE_mean']).to_string(index=False))
print('Saved outputs in',out)





c:\Users\Usuario\AppData\Local\Programs\Python\Python312\Lib\site-packages\joblib\externals\loky\backend\context.py:136: UserWarning: Could not find the number of physical cores for the following reason:
[WinError 2] O sistema não pode encontrar o arquivo especificado
Returning the number of logical cores instead. You can silence this warning by setting LOKY_MAX_CPU_COUNT to the number of cores you want to use.
  warnings.warn(
  File "c:\Users\Usuario\AppData\Local\Programs\Python\Python312\Lib\site-packages\joblib\externals\loky\backend\context.py", line 257, in _count_physical_cores
    cpu_info = subprocess.run(
               ^^^^^^^^^^^^^^^
  File "c:\Users\Usuario\AppData\Local\Programs\Python\Python312\Lib\subprocess.py", line 548, in run
    with Popen(*popenargs, **kwargs) as process:
         ^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "c:\Users\Usuario\AppData\Local\Programs\Python\Python312\Lib\subprocess.py", line 1026, in __init__
    self._execute_child(args, executable, preexec

Overall results: positive gain_vs_ewm3 means residual model beats EWM3:
     target  share_high segmento                     modelo  folds  RMSE_mean  MAE_mean  R2_mean  gain_vs_ewm3_pct  gain_vs_mean_pct  folds_better_than_ewm3
RISCO_CRIME        0.20    todos    ridgeresid_hurdle_top20      3   0.020208  0.011120 0.987136          0.682776          9.446444                       2
RISCO_CRIME        0.20    todos       rfresid_hurdle_top20      3   0.020223  0.011101 0.987117          0.604169          9.374052                       2
RISCO_CRIME        0.05    todos                ewm3_global      3   0.020363  0.011110 0.986917          0.000000          8.824851                       0
RISCO_CRIME        0.10    todos                ewm3_global      3   0.020363  0.011110 0.986917          0.000000          8.824851                       0
RISCO_CRIME        0.20    todos                ewm3_global      3   0.020363  0.011110 0.986917          0.000000          8.824851           